# PI-RADS 3 radiomics pipeline

This notebook runs the radiomics analysis step by step.

## Research aim

We compare the usual **PSA density (PSAD) baseline** with models based on prostate MRI radiomics. The main question is not only whether a radiomics model performs well internally, but also whether its performance transfers to a different dataset or institution.

## Pipeline overview

1. Define and inspect the PI-RADS 3 clinical cohort.
2. Read the T2W and ADC MRI scans and whole-gland masks.
3. Extract first-order and texture radiomics features.
4. Join radiomics features to clinical outcomes using patient/study IDs.
5. Compare PSAD, clinical, radiomics, radiomics + PSAD, and random-forest models.
6. Evaluate internal cross-validation and, when available, external validation.

**Important:** the demo data are artificial and only test that the software pipeline works. Their performance numbers must not be interpreted as medical results.

## 0. Setup

This notebook must run in the project's virtual environment (`.venv`), never in the system Python.

1. One time, in a terminal in the project folder: `python3 -m venv .venv`
2. In VS Code, click **Select Kernel** (top right) → **Python Environments** → `.venv`.
3. Run the cell below. It installs everything listed in `requirements.txt` into that venv. PyRadiomics is installed from GitHub because the normal PyPI package fails to build on Python 3.13.

If the cell installed anything new, restart the kernel before continuing.

In [1]:
import sys

# Safety check: refuse to install packages outside a virtual environment.
if sys.prefix == sys.base_prefix:
    raise RuntimeError(
        f"This kernel is not a virtual environment ({sys.executable}). "
        "Select the .venv kernel first so packages are not installed system-wide."
    )
print(f"Kernel Python: {sys.executable}")

%pip install -q -r requirements.txt

Kernel Python: /Users/martijnengel/Downloads/radiomics/.venv/bin/python
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import psad_baseline as pb
import extract_radiomics as er
import compare_models as cm

pd.set_option("display.max_columns", 20)
PROJECT_DIR = Path.cwd()
print(f"Project directory: {PROJECT_DIR}")
print(f"Target sensitivity: {pb.TARGET_SENSITIVITY:.0%}")

Project directory: /Users/martijnengel/Downloads/radiomics
Target sensitivity: 90%


### Choose the run mode

- `DEMO = False` (default) uses the real PI-CAI data:
  - MRI scans (`.mha`) in `picai_public_images_fold/<patient_id>/`
  - whole-gland masks in `picai_labels/anatomical_delineations/whole_gland/AI/Bosma22b/`
  - clinical table `picai_labels/clinical_information/marksheet.csv`
- `DEMO = True` uses synthetic data, to check the software after an installation.

The folder names are set in `PATHS` in `extract_radiomics.py` and `MAIN_DATASET` in `psad_baseline.py`.

In [3]:
DEMO = False
print("Running with synthetic data" if DEMO else "Running with real PI-CAI data")

Running with real PI-CAI data


## 1. Define the clinical cohort

The baseline script standardises the dataset-specific column names and applies the same eligibility rules everywhere. In this project, the cohort is restricted to PI-RADS 3 patients with an outcome and valid PSAD.

For real PI-CAI data, inspect and update `MAIN_DATASET["columns"]` in `psad_baseline.py` before running this cell. The printed counts should be recorded as part of the data description.

In [4]:
if DEMO:
    clinical = pb.standardise(
        pb.make_fake_data(300, seed=0),
        pb.MAIN_DATASET,
    )
else:
    clinical = pb.load_dataset(pb.MAIN_DATASET)

print(f"Final cohort size: {len(clinical)}")
print(f"csPCa-positive patients: {clinical['label'].sum()} ({clinical['label'].mean():.1%})")
display(clinical.head())

[PI-CAI] patients in file: 1500
[PI-CAI] PI-RADS 3 (highest lesion = 3): 151
[PI-CAI] ...with biopsy/surgery result: 151
[PI-CAI] ...with PSAD and outcome available: 149 (csPCa: 18, 12%)

Final cohort size: 149
csPCa-positive patients: 18 (12.1%)


,patient_id,study_id,pirads,psad,age,label,histopath,center
4,10004,1000004,3.0,0.100000,67,0,SysBx+MRBx,RUMC
7,10007,1000007,3.0,0.090000,68,0,SysBx+MRBx,RUMC
15,10015,1000015,3.0,0.460000,64,0,MRBx,PCNN
18,10018,1000018,3.0,0.251429,70,0,MRBx,PCNN
27,10027,1000027,3.0,0.200000,63,0,MRBx,PCNN


The identifiers are essential: they are the link between the clinical table and the MRI-derived features. Before extracting features, check that they are unique at the patient/study level and that there are no missing values.

In [5]:
id_columns = ["patient_id", "study_id"]
print("Missing identifiers:")
display(clinical[id_columns].isna().sum().to_frame("missing"))
print(f"Duplicate patient/study pairs: {clinical.duplicated(id_columns).sum()}")
print("Outcome counts:")
display(clinical["label"].value_counts().rename({0: "no csPCa", 1: "csPCa"}).to_frame("patients"))

Missing identifiers:


,missing
patient_id,0
study_id,0


Duplicate patient/study pairs: 0
Outcome counts:


,patients
label,
no csPCa,131
csPCa,18


### Which patients have images on disk?

The PI-CAI images are downloaded separately from the clinical table (5 folds, ~300 patients each), so usually only part of the cohort has images. A patient can only be used if the T2W scan, the ADC scan **and** the whole-gland mask are all present.

In [6]:
def has_all_files(row, paths):
    """True if the T2W, ADC and whole-gland mask files all exist for this patient."""
    ids = {"patient_id": row["patient_id"], "study_id": row["study_id"]}
    patterns = list(paths["images"].values()) + [paths["mask"]]
    return all(os.path.exists(pattern.format(**ids)) for pattern in patterns)

if not DEMO:
    paths = er.PATHS["main"]
    image_folder = Path(paths["images"]["t2w"]).parts[0]
    image_patients = [p.name for p in Path(image_folder).iterdir() if p.is_dir()]
    on_disk = clinical.apply(has_all_files, axis=1, paths=paths).astype(bool)
    print(f"Patient folders in {image_folder}/: {len(image_patients)}")
    print(f"PI-RADS 3 cohort patients with T2W + ADC + gland mask: {on_disk.sum()} of {len(clinical)}")

Patient folders in picai_public_images_fold/: 1476
PI-RADS 3 cohort patients with T2W + ADC + gland mask: 149 of 149


### Technical check on a few downloaded images

This extracts features from a small sample of downloaded patients (`IMAGE_CHECK_N`), whatever their PI-RADS score, and saves them to a separate file (`radiomics_image_check.csv`). It only shows that the real scans and masks can be read and lined up. These patients are not the study cohort and are not used for modelling.

In [7]:
IMAGE_CHECK_N = 5   # number of downloaded patients to test; extraction takes a few seconds each

if not DEMO:
    marksheet = pd.read_csv(pb.MAIN_DATASET["file"], dtype=str)
    downloaded = marksheet[marksheet.apply(has_all_files, axis=1, paths=paths).astype(bool)]
    print(f"Downloaded patients (any PI-RADS) with T2W + ADC + gland mask: {len(downloaded)}")
    sample = downloaded.head(IMAGE_CHECK_N)
    image_check = er.extract_all(sample, dict(paths, output="radiomics_image_check.csv"))
    display(sample[["patient_id", "study_id", "lesion_PIRADS", "case_csPCa"]]
            .merge(image_check[["patient_id", "t2w_firstorder_Mean", "adc_firstorder_Mean"]],
                   on="patient_id", how="left"))

Downloaded patients (any PI-RADS) with T2W + ADC + gland mask: 1500
  processed 5/5 patients

Saved 5 patients x 84 features -> radiomics_image_check.csv


,patient_id,study_id,lesion_PIRADS,case_csPCa,t2w_firstorder_Mean,adc_firstorder_Mean
0,10000,1000000,4,NO,-18.514086,1101.222666
1,10001,1000001,2,NO,-13.554938,1451.032631
2,10002,1000002,2,NO,8.010691,1214.797973
3,10003,1000003,2,NO,5.911029,1452.953283
4,10004,1000004,"3,2",NO,-41.300629,1401.391150


## 2. Extract radiomics from MRI

For each patient, the extraction script reads:

- a T2-weighted image;
- an ADC image;
- a whole-gland prostate mask.

The mask is resampled to each image grid using nearest-neighbour interpolation. We extract first-order statistics and GLCM texture features, but not shape features because prostate size is already represented clinically by volume and PSAD.

In [8]:
if DEMO:
    demo_patients = clinical.head(40).copy()
    demo_paths = er.make_fake_images(demo_patients, folder="demo_images")
    radiomics = er.extract_all(demo_patients, demo_paths)
else:
    # Only the cohort patients whose files were found in the check above.
    radiomics = er.extract_all(clinical[on_disk], paths)

print(f"Radiomics table shape: {radiomics.shape}")
display(radiomics.head())

  processed 10/149 patients
  processed 20/149 patients
  processed 30/149 patients
  processed 40/149 patients
  processed 50/149 patients
  processed 60/149 patients
  processed 70/149 patients
  processed 80/149 patients
  processed 90/149 patients
  processed 100/149 patients
  processed 110/149 patients
  processed 120/149 patients
  processed 130/149 patients
  processed 140/149 patients
  processed 149/149 patients

Saved 149 patients x 84 features -> radiomics_picai.csv
Radiomics table shape: (149, 86)


,patient_id,study_id,t2w_firstorder_10Percentile,t2w_firstorder_90Percentile,t2w_firstorder_Energy,t2w_firstorder_Entropy,t2w_firstorder_InterquartileRange,t2w_firstorder_Kurtosis,t2w_firstorder_Maximum,t2w_firstorder_MeanAbsoluteDeviation,...,adc_glcm_Imc2,adc_glcm_InverseVariance,adc_glcm_JointAverage,adc_glcm_JointEnergy,adc_glcm_JointEntropy,adc_glcm_MCC,adc_glcm_MaximumProbability,adc_glcm_SumAverage,adc_glcm_SumEntropy,adc_glcm_SumSquares
0,10004,1000004,-89.237209,19.177998,6.794151e+08,5.079199,53.444116,4.266291,281.817654,33.964083,...,0.652226,0.223332,16.401715,0.003373,8.631529,0.468873,0.009414,32.803431,5.228533,29.122060
1,10007,1000007,-96.303833,71.216428,4.908429e+08,5.593256,95.725863,2.346809,226.770955,52.210914,...,0.817340,0.233691,19.122593,0.003080,8.716841,0.675046,0.007861,38.245186,5.470732,38.178757
2,10015,1000015,12.572546,161.465591,1.614431e+09,5.433351,78.364761,3.101993,297.297843,46.337672,...,0.779348,0.109672,57.538650,0.001077,10.813181,0.571804,0.018780,115.077299,6.525353,185.393218
3,10018,1000018,-57.847246,109.324515,1.171881e+08,5.654382,91.819714,2.377236,222.602097,51.377072,...,0.728569,0.263733,16.183918,0.005020,8.022172,0.567406,0.011942,32.367837,5.021037,20.940111
4,10027,1000027,-47.210468,48.138987,4.377168e+07,4.941875,45.446937,4.764418,213.886638,30.153167,...,0.656245,0.260687,24.761375,0.004637,8.357800,0.523007,0.010110,49.522750,5.159805,28.031303


A successful extraction should produce one row per successfully processed patient, two identifier columns, and many feature columns prefixed with `t2w_` or `adc_`. A smaller-than-expected table means that some images or masks could not be read and should be investigated before modelling.

In [9]:
radiomic_columns = [
    column for column in radiomics.columns
    if column.startswith(("t2w_", "adc_"))
]
print(f"Patients with radiomics: {len(radiomics)}")
print(f"Radiomics features: {len(radiomic_columns)}")
print("Features by sequence:")
display(pd.Series({
    "T2W": sum(column.startswith("t2w_") for column in radiomic_columns),
    "ADC": sum(column.startswith("adc_") for column in radiomic_columns),
}))

Patients with radiomics: 149
Radiomics features: 84
Features by sequence:


T2W    42
ADC    42
dtype: int64

The synthetic images deliberately make ADC lower in positive cases. This is only a pipeline sanity check. A visible difference here confirms that the mask and image grids were handled, not that radiomics is clinically useful.

In [10]:
if DEMO and "adc_firstorder_Mean" in radiomics.columns:
    sanity = radiomics.merge(clinical[["patient_id", "label"]], on="patient_id")
    display(sanity.groupby("label")["adc_firstorder_Mean"].mean().rename({0: "no csPCa", 1: "csPCa"}).to_frame("mean ADC"))

### Enough patients for modelling?

The models use stratified 5-fold cross-validation, so every fold needs at least one csPCa and one non-csPCa patient. The next cell stops the notebook with an explanation if there are too few patients. In that case, download more PI-CAI image folds into `picai_public_images_fold/` together with their whole-gland masks.

In [11]:
if len(radiomics):
    labelled = clinical.merge(radiomics[["patient_id", "study_id"]], on=["patient_id", "study_id"])
else:
    labelled = clinical.iloc[0:0]
n_pos = int(labelled["label"].sum())
n_neg = len(labelled) - n_pos
print(f"Cohort patients with radiomics: {len(labelled)} (csPCa: {n_pos}, no csPCa: {n_neg})")
if min(n_pos, n_neg) < cm.N_FOLDS:
    raise RuntimeError(
        f"Not enough patients for {cm.N_FOLDS}-fold cross-validation: need at least "
        f"{cm.N_FOLDS} csPCa and {cm.N_FOLDS} non-csPCa patients with images, "
        f"found {n_pos} and {n_neg}. Download more PI-CAI image folds and masks."
    )

Cohort patients with radiomics: 149 (csPCa: 18, no csPCa: 131)


## 3. Join clinical outcomes and radiomics

The join is a critical quality-control step. Patients without both clinical data and radiomics are not silently treated as negative cases; they are excluded from this modelling table and counted explicitly.

In [12]:
data = cm.join(clinical, radiomics, "demo" if DEMO else pb.MAIN_DATASET["name"])
print(f"Final modelling table: {len(data)} patients")
print(f"Positive cases in modelling table: {data['label'].sum()}")
display(data[["patient_id", "study_id", "psad", "age", "label"] + radiomic_columns[:4]].head())

[PI-CAI] patients with clinical data AND radiomics: 149 (of 149; csPCa: 18)
Final modelling table: 149 patients
Positive cases in modelling table: 18


,patient_id,study_id,psad,age,label,t2w_firstorder_10Percentile,t2w_firstorder_90Percentile,t2w_firstorder_Energy,t2w_firstorder_Entropy
0,10004,1000004,0.100000,67,0,-89.237209,19.177998,6.794151e+08,5.079199
1,10007,1000007,0.090000,68,0,-96.303833,71.216428,4.908429e+08,5.593256
2,10015,1000015,0.460000,64,0,12.572546,161.465591,1.614431e+09,5.433351
3,10018,1000018,0.251429,70,0,-57.847246,109.324515,1.171881e+08,5.654382
4,10027,1000027,0.200000,63,0,-47.210468,48.138987,4.377168e+07,4.941875


## 4. Compare the models

The models are evaluated with stratified 5-fold cross-validation. Within each training fold:

- feature scaling is learned only from the training patients;
- LASSO feature selection is learned only from the training patients;
- the biopsy threshold is chosen to target 90% sensitivity.

This prevents information from the test fold leaking into model training.

In [13]:
radiomic_columns = [
    column for column in radiomics.columns
    if column.startswith(("t2w_", "adc_"))
]
clinical_columns = ["psad"]
if data["age"].notna().all():
    clinical_columns.append("age")

models = cm.model_list(radiomic_columns, clinical_columns)
internal_results = cm.cross_validate(data, models)
display(internal_results.round(3))
internal_results.to_csv("results_internal_notebook.csv")
print("Saved results_internal_notebook.csv")

  done: 1. PSAD rule
  done: 2. Clinical (psad + age)


/Users/martijnengel/Downloads/radiomics/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/martijnengel/Downloads/radiomics/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:2102: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
/Users/martijnengel/Downloads/radiomics/.venv/lib/python3.13/site-packages/sklear

  done: 3. Radiomics only


/Users/martijnengel/Downloads/radiomics/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/martijnengel/Downloads/radiomics/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:2102: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
/Users/martijnengel/Downloads/radiomics/.venv/lib/python3.13/site-packages/sklear

  done: 4. Radiomics + PSAD
  done: 5. Random forest (radiomics + PSAD)


,sensitivity,specificity,NPV,biopsies_avoided_%,missed_csPCa,biopsy_yield,AUC,avoided_%_at_matched_sens
1. PSAD rule,0.833,0.198,0.897,19.463,3.0,0.125,0.618,14.094
2. Clinical (psad + age),0.944,0.046,0.857,4.698,1.0,0.120,0.507,8.725
3. Radiomics only,1.000,0.038,1.000,3.356,0.0,0.125,0.465,12.081
4. Radiomics + PSAD,1.000,0.092,1.000,8.054,0.0,0.131,0.471,15.436
5. Random forest (radiomics + PSAD),0.889,0.130,0.895,12.752,2.0,0.123,0.480,9.396


Saved results_internal_notebook.csv


### How to read the result table

- `AUC` measures ranking performance across thresholds.
- `sensitivity` measures the proportion of csPCa cases detected.
- `missed_csPCa` is the number of positive cases missed at the learned threshold.
- `biopsies_avoided_%` is the clinically relevant trade-off at the threshold learned from training data.
- `avoided_%_at_matched_sens` gives a same-sensitivity comparison, but its threshold is selected on the evaluation data and is therefore optimistic. Treat it as a supplementary comparison.

In [14]:
summary_columns = [
    "AUC", "sensitivity", "NPV", "missed_csPCa",
    "biopsies_avoided_%", "avoided_%_at_matched_sens"
]
available_summary_columns = [
    column for column in summary_columns if column in internal_results.columns
]
display(internal_results[available_summary_columns].round(3))

,AUC,sensitivity,NPV,missed_csPCa,biopsies_avoided_%,avoided_%_at_matched_sens
1. PSAD rule,0.618,0.833,0.897,3.0,19.463,14.094
2. Clinical (psad + age),0.507,0.944,0.857,1.0,4.698,8.725
3. Radiomics only,0.465,1.000,1.000,0.0,3.356,12.081
4. Radiomics + PSAD,0.471,1.000,1.000,0.0,8.054,15.436
5. Random forest (radiomics + PSAD),0.480,0.889,0.895,2.0,12.752,9.396


## 5. Inspect which features LASSO retained

The radiomics-only and radiomics + PSAD models use L1-regularised logistic regression. Coefficients equal to zero are discarded by the model. The selected features are useful for describing the model, but they should not automatically be interpreted as biological biomarkers: feature stability must be checked in a larger or repeated validation study.

In [15]:
fitted_models = {}
for name, (columns, model) in models.items():
    if model is None:
        fitted_models[name] = None
    else:
        fitted_models[name] = model.fit(data[columns], data["label"])
cm.show_selected_features(fitted_models, radiomic_columns)

/Users/martijnengel/Downloads/radiomics/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/martijnengel/Downloads/radiomics/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:2102: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
/Users/martijnengel/Downloads/radiomics/.venv/lib/python3.13/site-packages/sklear


3. Radiomics only: LASSO kept 7 of 84 features. Top 5:
t2w_glcm_Imc1                 -0.319
t2w_firstorder_10Percentile   -0.170
adc_firstorder_Minimum         0.155
t2w_glcm_MaximumProbability   -0.153
adc_firstorder_Energy         -0.102

4. Radiomics + PSAD: LASSO kept 8 of 85 features. Top 5:
psad                           0.394
t2w_glcm_Imc1                 -0.362
t2w_firstorder_10Percentile   -0.238
adc_firstorder_Minimum         0.186
t2w_glcm_MaximumProbability   -0.096


## 6. Optional external validation

When an AI4AR clinical table and radiomics table are available, configure `EXTERNAL_DATASET` in `psad_baseline.py`, extract AI4AR features with `extract_radiomics.py --dataset external`, and then run the following cell.

The external dataset must not be used to tune features, model hyperparameters, or thresholds. It is used only to test how well a model trained on PI-CAI transfers to another domain.

In [16]:
# Example for real external validation:
# external_clinical = pb.load_dataset(pb.EXTERNAL_DATASET)
# external_radiomics = pd.read_csv("radiomics_ai4ar.csv")
# external_data = cm.join(external_clinical, external_radiomics, pb.EXTERNAL_DATASET["name"])
# external_results, fitted = cm.external_validate(data, external_data, models)
# display(external_results.round(3))
# external_results.to_csv("results_external_notebook.csv")

## 7. Checklist before sharing results

- [ ] The real PI-CAI column names in `psad_baseline.py` were checked against the source file.
- [ ] The MRI and mask paths in `extract_radiomics.py` were verified on several patients.
- [ ] The number of patients lost during radiomics extraction was recorded.
- [ ] Patient/study IDs match between clinical and radiomics tables.
- [ ] The number of positive cases is reported alongside the total sample size.
- [ ] Scaling, feature selection, and threshold selection occur inside training data only.
- [ ] Synthetic demo results are clearly labelled as technical validation.
- [ ] External results are reported separately from internal cross-validation.
- [ ] Sensitivity and missed csPCa cases are interpreted before biopsies avoided.

The notebook is an orchestration and explanation layer. The reusable implementation remains in the three Python scripts, so changes to the modelling logic should be made there and then reflected here.